# OMEGA-SENTINEL — Complete Colab Research Prototype

**OMEGA-SENTINEL** combines temporal modeling, multi-agent graph reasoning, trajectory prediction, a lightweight latent world model, uncertainty estimation, counterfactual simulation, risk, recoverability, safety memory, and planning.

> **Data note:** this notebook runs end-to-end on a synthetic multi-agent driving dataset by default. The Waymo SDK/TensorFlow installation is intentionally not required. A real Waymo/Parquet adapter can be connected later without changing the model interfaces.

In [ ]:
!pip -q install numpy pandas matplotlib scikit-learn networkx tqdm

In [ ]:
import os, math, random, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

for p in [
    "omega/data", "omega/models", "omega/world_model",
    "omega/uncertainty", "omega/counterfactual",
    "omega/memory", "omega/planner", "omega/evaluation",
    "results", "checkpoints"
]:
    Path(p).mkdir(parents=True, exist_ok=True)

## 1. Configuration

In [ ]:
PAST = 20
FUTURE = 60
FEATURES = 7
TARGET_FEATURES = 4
N_AGENTS = 6
NUM_SCENES = 600
BATCH_SIZE = 32

# Feature order:
# x, y, vx, vy, heading, length, width

## 2. Synthetic multi-agent scene generator

In [ ]:
def make_scene(past=PAST, future=FUTURE, n_agents=N_AGENTS):
    T = past + future
    t = np.arange(T, dtype=np.float32)
    data = np.zeros((n_agents, T, FEATURES), dtype=np.float32)

    lane_offsets = np.linspace(-6.0, 6.0, n_agents)
    speeds = np.random.uniform(0.75, 1.35, n_agents)
    phases = np.random.uniform(0, 2*np.pi, n_agents)

    for a in range(n_agents):
        x = speeds[a] * t + 1.0 * np.sin(0.045*t + phases[a])
        y = lane_offsets[a] + 0.8 * np.sin(0.035*t + phases[a])

        x += np.random.normal(0, 0.04, T)
        y += np.random.normal(0, 0.03, T)

        vx = np.gradient(x)
        vy = np.gradient(y)
        heading = np.arctan2(vy, vx)

        data[a, :, 0] = x
        data[a, :, 1] = y
        data[a, :, 2] = vx
        data[a, :, 3] = vy
        data[a, :, 4] = heading
        data[a, :, 5] = np.random.uniform(4.0, 5.2)
        data[a, :, 6] = np.random.uniform(1.7, 2.2)

    return data.astype(np.float32)


def build_synthetic_scenes(n=NUM_SCENES):
    return [make_scene() for _ in range(n)]


scenes = build_synthetic_scenes()
print("Scenes:", len(scenes))
print("Single scene:", scenes[0].shape)

In [ ]:
scene = scenes[0]

plt.figure(figsize=(11, 6))
for agent in range(N_AGENTS):
    plt.plot(scene[agent, :, 0], scene[agent, :, 1], alpha=0.8)
    plt.scatter(scene[agent, PAST-1, 0], scene[agent, PAST-1, 1], s=35)

plt.axvline(scene[0, PAST-1, 0], linestyle="--", alpha=0.4)
plt.title("OMEGA-SENTINEL Multi-Agent Scene")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(alpha=0.2)
plt.show()

## 3. Dataset and scene graph construction

In [ ]:
def build_edges(node_features, radius=30.0):
    pos = node_features[:, :2]
    edges = []

    for i in range(len(pos)):
        for j in range(len(pos)):
            if i != j and torch.linalg.vector_norm(pos[i] - pos[j]) < radius:
                edges.append([i, j])

    if not edges:
        edges = [[0, 0]]

    return torch.tensor(edges, dtype=torch.long, device=node_features.device).t().contiguous()


class OMEGADataset(Dataset):
    def __init__(self, scenes):
        self.scenes = scenes

    def __len__(self):
        return len(self.scenes)

    def __getitem__(self, idx):
        s = torch.tensor(self.scenes[idx], dtype=torch.float32)

        past = s[:, :PAST, :]
        future = s[:, PAST:, :4]

        return past, future


dataset = OMEGADataset(scenes)
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len

train_ds, val_ds = torch.utils.data.random_split(
    dataset,
    [train_len, val_len],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

past, future = dataset[0]
print("Past:", past.shape)
print("Future:", future.shape)

## 4. Metrics

In [ ]:
def displacement_error(pred, true):
    return torch.linalg.vector_norm(pred[..., :2] - true[..., :2], dim=-1)


def ADE(pred, true):
    return displacement_error(pred, true).mean().item()


def FDE(pred, true):
    return displacement_error(pred[:, -1], true[:, -1]).mean().item()


def regression_metrics(pred, true):
    diff = pred - true
    mse = torch.mean(diff ** 2).item()
    mae = torch.mean(torch.abs(diff)).item()
    rmse = math.sqrt(mse)
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "ADE": ADE(pred, true),
        "FDE": FDE(pred, true)
    }

## 5. LSTM baseline

In [ ]:
class TrajectoryLSTM(nn.Module):
    def __init__(self, input_dim=FEATURES, hidden=128, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden, layers,
            batch_first=True
        )
        self.head = nn.Linear(hidden, FUTURE * 4)

    def forward(self, x):
        h, _ = self.lstm(x)
        z = self.head(h[:, -1])
        return z.view(x.size(0), FUTURE, 4)


baseline = TrajectoryLSTM().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(baseline.parameters(), lr=1e-3)

for epoch in range(5):
    baseline.train()
    total = 0

    for past_b, future_b in train_loader:
        x = past_b[:, 0].to(DEVICE)
        y = future_b[:, 0].to(DEVICE)

        pred = baseline(x)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1}/5 | loss={total/len(train_loader):.5f}")

## 6. Temporal Transformer

In [ ]:
class TemporalTransformer(nn.Module):
    def __init__(self, input_dim=FEATURES, d_model=128, heads=8, layers=3):
        super().__init__()

        self.projection = nn.Linear(input_dim, d_model)
        self.position = nn.Parameter(
            torch.zeros(1, PAST, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=heads,
            dim_feedforward=4*d_model,
            dropout=0.1,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=layers
        )

    def forward(self, x):
        z = self.projection(x)
        z = z + self.position[:, :x.size(1)]
        z = self.encoder(z)
        return z[:, -1]

## 7. Scene GAT — pure PyTorch implementation

In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.attn = nn.Linear(2*out_dim, 1, bias=False)

    def forward(self, x, edge_index):
        h = self.W(x)
        src, dst = edge_index

        score = self.attn(
            torch.cat([h[src], h[dst]], dim=-1)
        ).squeeze(-1)

        score = F.leaky_relu(score, 0.2)

        out = torch.zeros_like(h)

        for node in torch.unique(dst):
            mask = dst == node
            alpha = torch.softmax(score[mask], dim=0)
            out[node] += torch.sum(
                alpha.unsqueeze(-1) * h[src[mask]],
                dim=0
            )

        return out


class SceneGAT(nn.Module):
    def __init__(self, input_dim=FEATURES, hidden=64, output=128):
        super().__init__()
        self.gat1 = GATLayer(input_dim, hidden)
        self.gat2 = GATLayer(hidden, output)

    def forward(self, x, edge_index):
        x = F.elu(self.gat1(x, edge_index))
        x = self.gat2(x, edge_index)
        return x

## 8. OMEGA Fusion

In [ ]:
class OMEGAFusion(nn.Module):
    def __init__(self):
        super().__init__()

        self.temporal = TemporalTransformer()
        self.gat = SceneGAT()

        self.fusion = nn.Sequential(
            nn.Linear(128 + 128, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10)
        )

        self.predictor = nn.Linear(
            256,
            FUTURE * 4
        )

    def forward(
        self,
        trajectory,
        node_features,
        edge_index,
        target_index=0
    ):
        temporal_embedding = self.temporal(trajectory)

        node_embeddings = self.gat(
            node_features,
            edge_index
        )

        social_embedding = node_embeddings[target_index]

        # Correctly broadcast one scene embedding over the batch.
        social_embedding = social_embedding.unsqueeze(0).expand(
            trajectory.size(0), -1
        )

        combined = torch.cat(
            [temporal_embedding, social_embedding],
            dim=1
        )

        fused = self.fusion(combined)

        prediction = self.predictor(fused)

        return prediction.view(
            trajectory.size(0),
            FUTURE,
            4
        )

In [ ]:
omega = OMEGAFusion().to(DEVICE)

test_scene = torch.tensor(
    scenes[0],
    dtype=torch.float32,
    device=DEVICE
)

test_nodes = test_scene[:, PAST-1, :]
test_edges = build_edges(test_nodes)

test_trajectory = test_scene[0, :PAST, :].unsqueeze(0)

test_output = omega(
    test_trajectory,
    test_nodes,
    test_edges
)

print("Input:", test_trajectory.shape)
print("Nodes:", test_nodes.shape)
print("Edges:", test_edges.shape)
print("Output:", test_output.shape)

assert test_output.shape == (1, FUTURE, 4)
print("OMEGA Fusion shape test PASSED.")

## 9. Train OMEGA Fusion

In [ ]:
omega = OMEGAFusion().to(DEVICE)
optimizer = torch.optim.AdamW(
    omega.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

omega_history = []

for epoch in range(5):
    omega.train()
    total = 0

    for past_b, future_b in train_loader:
        past_b = past_b.to(DEVICE)
        future_b = future_b.to(DEVICE)

        batch_losses = []

        for b in range(past_b.size(0)):
            trajectory = past_b[b, 0].unsqueeze(0)
            nodes = past_b[b, :, -1, :]
            edges = build_edges(nodes)

            pred = omega(
                trajectory,
                nodes,
                edges,
                target_index=0
            )

            target = future_b[b, 0].unsqueeze(0)
            batch_losses.append(criterion(pred, target))

        loss = torch.stack(batch_losses).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(omega.parameters(), 1.0)
        optimizer.step()

        total += loss.item()

    epoch_loss = total / len(train_loader)
    omega_history.append(epoch_loss)
    print(f"Epoch {epoch+1}/5 | loss={epoch_loss:.5f}")

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(omega_history, marker="o")
plt.title("OMEGA Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=.2)
plt.show()

## 10. OMEGA evaluation

In [ ]:
omega.eval()
predictions = []
targets = []

with torch.no_grad():
    for past_b, future_b in val_loader:
        past_b = past_b.to(DEVICE)
        future_b = future_b.to(DEVICE)

        for b in range(past_b.size(0)):
            trajectory = past_b[b, 0].unsqueeze(0)
            nodes = past_b[b, :, -1, :]
            edges = build_edges(nodes)

            pred = omega(
                trajectory,
                nodes,
                edges
            )

            predictions.append(pred[0])
            targets.append(future_b[b, 0].to(DEVICE))

predictions = torch.stack(predictions)
targets = torch.stack(targets)

metrics = regression_metrics(predictions, targets)
print(json.dumps(metrics, indent=2))

In [ ]:
i = 0

plt.figure(figsize=(10,6))
plt.plot(
    targets[i,:,0].cpu(),
    targets[i,:,1].cpu(),
    label="Ground Truth"
)
plt.plot(
    predictions[i,:,0].cpu(),
    predictions[i,:,1].cpu(),
    label="OMEGA Prediction"
)
plt.scatter(
    predictions[i,0,0].cpu(),
    predictions[i,0,1].cpu(),
    s=60
)
plt.title("OMEGA Future Trajectory")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 11. Latent World Model

In [ ]:
class WorldEncoder(nn.Module):
    def __init__(self, input_dim=4, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Linear(128, latent_dim)
        )

    def forward(self, x):
        return self.net(x)


class WorldDynamics(nn.Module):
    def __init__(self, latent_dim=128, action_dim=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + action_dim, 256),
            nn.GELU(),
            nn.Linear(256, latent_dim)
        )

    def forward(self, z, action):
        return self.net(torch.cat([z, action], dim=-1))


class WorldDecoder(nn.Module):
    def __init__(self, latent_dim=128, output_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.GELU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, z):
        return self.net(z)


class OmegaWorldModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = WorldEncoder()
        self.dynamics = WorldDynamics()
        self.decoder = WorldDecoder()

    def step(self, state, action):
        z = self.encoder(state)
        z_next = self.dynamics(z, action)
        return z_next, self.decoder(z_next)

    def rollout(self, state, actions):
        z = self.encoder(state)
        outputs = []

        for action in actions:
            z = self.dynamics(z, action)
            outputs.append(self.decoder(z))

        return torch.stack(outputs, dim=1)


world_model = OmegaWorldModel().to(DEVICE)

state = torch.randn(1, 4, device=DEVICE)
action = F.one_hot(
    torch.tensor([0], device=DEVICE),
    num_classes=5
).float()

z, next_state = world_model.step(state, action)

print("Latent:", z.shape)
print("Predicted state:", next_state.shape)

## 12. Monte Carlo uncertainty

In [ ]:
class UncertaintyHead(nn.Module):
    def __init__(self, input_dim=4, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.GELU(),
            nn.Dropout(.20),
            nn.Linear(hidden, 128),
            nn.GELU(),
            nn.Dropout(.20),
            nn.Linear(128, 8)
        )

    def forward(self, x):
        out = self.net(x)
        mean = out[..., :4]
        log_var = torch.clamp(out[..., 4:], -6, 4)
        return mean, log_var


uncertainty_head = UncertaintyHead().to(DEVICE)

def mc_uncertainty(model, x, runs=20):
    model.train()
    samples = []

    with torch.no_grad():
        for _ in range(runs):
            samples.append(model(x)[0])

    samples = torch.stack(samples)
    return samples.mean(0), samples.std(0)


sample_state = torch.randn(1, 4, device=DEVICE)
mean_u, std_u = mc_uncertainty(uncertainty_head, sample_state)

print("Mean:", mean_u.shape)
print("Std:", std_u.shape)

## 13. Counterfactual simulator

In [ ]:
ACTIONS = [
    "MAINTAIN",
    "ACCELERATE",
    "BRAKE",
    "LANE_LEFT",
    "LANE_RIGHT"
]

ACTION_INDEX = {name:i for i,name in enumerate(ACTIONS)}

def action_vector(name, device=DEVICE):
    return F.one_hot(
        torch.tensor([ACTION_INDEX[name]], device=device),
        num_classes=len(ACTIONS)
    ).float()


class CounterfactualSimulator:
    def __init__(self, world_model):
        self.world_model = world_model

    @torch.no_grad()
    def simulate(self, state, action_name, horizon=20):
        z = self.world_model.encoder(state)
        action = action_vector(action_name).to(state.device)
        outputs = []

        for _ in range(horizon):
            z = self.world_model.dynamics(z, action)
            outputs.append(self.world_model.decoder(z))

        return torch.stack(outputs, dim=1)


simulator = CounterfactualSimulator(world_model)
cf = simulator.simulate(
    torch.randn(1,4,device=DEVICE),
    "BRAKE"
)

print("Counterfactual shape:", cf.shape)

## 14. Interpretable risk metrics

In [ ]:
def distance_risk(ego, others):
    d = torch.linalg.vector_norm(
        others[..., :2] - ego[..., :2],
        dim=-1
    )
    return torch.exp(-d / 5.0).mean().item()


def minimum_distance(ego, others):
    d = torch.linalg.vector_norm(
        others[..., :2] - ego[..., :2],
        dim=-1
    )
    return float(d.min().item())


def time_to_collision(ego, other):
    relative_position = other[:2] - ego[:2]
    relative_velocity = other[2:4] - ego[2:4]

    closing = -torch.dot(
        relative_position,
        relative_velocity
    ) / (torch.sum(relative_velocity**2) + 1e-6)

    return max(float(closing.item()), 0.0)


def combined_risk(ego_future, others_future, uncertainty=0.0):
    ego = ego_future[:, :2]
    others = others_future[:, :, :2]

    distances = torch.linalg.vector_norm(
        others - ego.unsqueeze(1),
        dim=-1
    )

    proximity = torch.exp(-distances / 5.0).mean()
    collision_component = torch.relu(2.5 - distances.min()) / 2.5

    return float(
        proximity.item()
        + collision_component.item()
        + uncertainty
    )

## 15. Recoverability

In [ ]:
def recoverability_score(ego_state, others_state):
    safe = 0

    # Lightweight local recovery test.
    for action in ACTIONS:
        future = simulator.simulate(
            ego_state.unsqueeze(0),
            action,
            horizon=10
        )[0]

        ego_xy = future[:, :2]
        other_xy = others_state[:10, :, :2]

        d = torch.linalg.vector_norm(
            ego_xy.unsqueeze(1) - other_xy,
            dim=-1
        )

        if float(d.min()) > 2.5:
            safe += 1

    return safe / len(ACTIONS)

## 16. Persistent Safety Memory

In [ ]:
class SafetyMemory:
    def __init__(self):
        self.items = []

    def add(
        self,
        embedding,
        risk,
        uncertainty,
        action,
        outcome,
        recoverability
    ):
        self.items.append({
            "embedding": embedding.detach().cpu(),
            "risk": float(risk),
            "uncertainty": float(uncertainty),
            "action": action,
            "outcome": outcome,
            "recoverability": float(recoverability)
        })

    def query(self, embedding, k=3):
        if not self.items:
            return []

        q = F.normalize(
            embedding.detach().cpu().flatten().unsqueeze(0),
            dim=-1
        )

        scored = []

        for item in self.items:
            e = F.normalize(
                item["embedding"].flatten().unsqueeze(0),
                dim=-1
            )
            score = float((q * e).sum())
            scored.append((score, item))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [item for _, item in scored[:k]]

    def save(self, path="results/safety_memory.pt"):
        torch.save(self.items, path)

    def load(self, path="results/safety_memory.pt"):
        self.items = torch.load(path, weights_only=False)


memory = SafetyMemory()

## 17. Risk-aware planning

In [ ]:
class RiskAwarePlanner:
    def __init__(self, memory):
        self.memory = memory

    def evaluate_actions(self, state, others):
        rows = []

        for action in ACTIONS:
            future = simulator.simulate(
                state.unsqueeze(0),
                action,
                horizon=min(FUTURE, 20)
            )[0]

            other_future = others[:future.size(0)]

            risk = combined_risk(
                future,
                other_future
            )

            recoverability = recoverability_score(
                state,
                others
            )

            score = risk + (1.0 - recoverability)

            rows.append({
                "action": action,
                "risk": risk,
                "recoverability": recoverability,
                "score": score
            })

        return pd.DataFrame(rows).sort_values(
            "score"
        ).reset_index(drop=True)

    def plan(self, state, others):
        table = self.evaluate_actions(state, others)
        return table.iloc[0]["action"], table


planner = RiskAwarePlanner(memory)

## 18. Final counterfactual planning demonstration

In [ ]:
demo_scene = torch.tensor(
    scenes[0],
    dtype=torch.float32,
    device=DEVICE
)

ego = demo_scene[0, PAST-1, :4]
others = demo_scene[1:, PAST-1:, :4]

chosen_action, planning_table = planner.plan(
    ego,
    others
)

print("Selected action:", chosen_action)
display(planning_table)

In [ ]:
plt.figure(figsize=(11, 7))

for agent in range(N_AGENTS):
    plt.plot(
        scene[agent, :, 0],
        scene[agent, :, 1],
        alpha=.25
    )

for action in ACTIONS:
    future = simulator.simulate(
        ego.unsqueeze(0),
        action,
        horizon=20
    )[0].cpu().numpy()

    plt.plot(
        future[:,0],
        future[:,1],
        label=action
    )

plt.scatter(
    float(ego[0]),
    float(ego[1]),
    s=90,
    marker="o",
    label="Ego"
)

plt.title("OMEGA-SENTINEL Counterfactual Futures")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 19. Grounded explanation layer

In [ ]:
def explain_decision(action, table):
    row = table[table["action"] == action].iloc[0]

    return (
        f"The planner selected {action}. "
        f"The measured planning score was {row['score']:.3f}, "
        f"with estimated risk {row['risk']:.3f} and "
        f"recoverability {row['recoverability']:.3f}. "
        "This explanation is grounded in the structured outputs "
        "of the current planner."
    )


print(explain_decision(chosen_action, planning_table))

## 20. Lightweight PPO components

In [ ]:
class PPOActorCritic(nn.Module):
    def __init__(self, state_dim=4, action_dim=5):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh()
        )

        self.actor = nn.Linear(128, action_dim)
        self.critic = nn.Linear(128, 1)

    def forward(self, state):
        z = self.body(state)
        return self.actor(z), self.critic(z).squeeze(-1)


ppo = PPOActorCritic().to(DEVICE)

logits, value = ppo(
    torch.randn(8,4,device=DEVICE)
)

print("PPO logits:", logits.shape)
print("PPO value:", value.shape)

## 21. PPO loss function

In [ ]:
def ppo_loss(
    logits,
    actions,
    old_log_probs,
    advantages,
    returns,
    values,
    clip_eps=0.2
):
    dist = torch.distributions.Categorical(logits=logits)
    log_probs = dist.log_prob(actions)

    ratio = torch.exp(log_probs - old_log_probs)

    clipped = torch.clamp(
        ratio,
        1 - clip_eps,
        1 + clip_eps
    )

    actor_loss = -torch.min(
        ratio * advantages,
        clipped * advantages
    ).mean()

    critic_loss = F.mse_loss(values, returns)

    entropy = dist.entropy().mean()

    return actor_loss + .5 * critic_loss - .01 * entropy

## 22. Save the research artifacts

In [ ]:
torch.save(
    {
        "model_state_dict": omega.state_dict(),
        "config": {
            "past": PAST,
            "future": FUTURE,
            "features": FEATURES
        },
        "metrics": metrics
    },
    "checkpoints/omega_fusion.pt"
)

pd.DataFrame([metrics]).to_csv(
    "results/omega_metrics.csv",
    index=False
)

planning_table.to_csv(
    "results/planner_actions.csv",
    index=False
)

memory.save()

print("Saved:")
print(" - checkpoints/omega_fusion.pt")
print(" - results/omega_metrics.csv")
print(" - results/planner_actions.csv")
print(" - results/safety_memory.pt")

# OMEGA-SENTINEL Architecture

```text
                   MULTI-AGENT SCENE
                          │
              ┌───────────┴───────────┐
              │                       │
       Temporal Transformer       Scene GAT
              │                       │
              └───────────┬───────────┘
                          │
                     OMEGA FUSION
                          │
                  Future Prediction
                          │
                     WORLD MODEL
                          │
          ┌───────────────┼───────────────┐
          │               │               │
     Uncertainty    Counterfactual   Safety Memory
          │               │               │
          └───────────────┼───────────────┘
                          │
                 Risk + Recoverability
                          │
                    MCTS / PPO
                          │
                       ACTION
                          │
                  GROUNDED EXPLANATION
```

### Research integrity

The default notebook uses **synthetic data** so it can run end-to-end. Do not report these synthetic results as Waymo results.

For the final paper, connect a verified Waymo Motion Dataset/Parquet data source, rerun the experiments, and report only measured results.

## Optional: Waymo Parquet adapter

In [ ]:
# This cell is intentionally optional.
# If you have downloaded Waymo v2 Parquet files, point DATA_PATH to them.
# No TensorFlow/Waymo SDK is required for this adapter.

import glob

def inspect_parquet(path):
    import pyarrow.parquet as pq
    table = pq.read_table(path)
    print("Columns:", table.column_names)
    print("Rows:", table.num_rows)
    return table

# Example:
# DATA_PATH = "/content/your_waymo_file.parquet"
# table = inspect_parquet(DATA_PATH)
#
# Build a project-specific column mapping after inspecting the real schema.